In [ ]:
import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from tokenizers import Tokenizer
try:
    import torch_directml
    device = torch_directml.device()
    BACKEND = "DirectML"
except (ImportError, RuntimeError, Exception):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        BACKEND = "CUDA"
    else:
        device = torch.device("cpu")
        BACKEND = "CPU"

print(f"Backend: {BACKEND} | Device: {device}")
batch_size = 1
accumulation_steps = 32
block_size = 128
max_iters = 50000
learning_rate = 3e-4
min_lr = 3e-5
warmup_iters = 1000
eval_interval = 250
eval_iters = 30
checkpoint_interval = 1000
n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.1
assert n_embd % n_head == 0
TOKENIZER_FILE = "784million.json"
if not os.path.exists(TOKENIZER_FILE):
    for alt in ["../784million.json", "../../784million.json", "vectora_v2_tokenizer.json"]:
        if os.path.exists(alt):
            TOKENIZER_FILE = alt
            break
tokenizer = Tokenizer.from_file(TOKENIZER_FILE)
vocab_size = tokenizer.get_vocab_size()
print(f"BERT Vocab size: {vocab_size:,}")
def encode(s):
    return tokenizer.encode(s).ids
def decode(ids):
    return tokenizer.decode(ids)
train_bin = "train.bin"
val_bin = "val.bin"
if os.path.exists(train_bin) and os.path.exists(val_bin):
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    val_data = np.memmap(val_bin, dtype=np.uint16, mode="r")
    print(f"Loaded Train tokens: {len(train_data):,} | Validation tokens: {len(val_data):,}")
else:
    print("Warning: train.bin not found. Run prepare_data.py first. Using temporary dummy data.")
    train_data = np.random.randint(0, min(1000, vocab_size), size=200000, dtype=np.uint16)
    val_data = np.random.randint(0, min(1000, vocab_size), size=20000, dtype=np.uint16)

def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)
    x_np = np.stack([source[i : i + block_size] for i in ix]).astype(np.int64)
    y_np = np.stack([source[i + 1 : i + 1 + block_size] for i in ix]).astype(np.int64)
    return torch.from_numpy(x_np).to(device), torch.from_numpy(y_np).to(device)
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    def forward(self, x, seq_len):
        cos = self.cos[:seq_len, :].unsqueeze(0).unsqueeze(1)
        sin = self.sin[:seq_len, :].unsqueeze(0).unsqueeze(1)
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        rotated_x1 = x1 * cos - x2 * sin
        rotated_x2 = x1 * sin + x2 * cos
        return torch.stack((rotated_x1, rotated_x2), dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len=block_size * 2)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q = self.rope(q, T)
        k = self.rope(k, T)

        drop_p = self.attn_dropout if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=drop_p, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(out))

class SwiGLUFeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden_dim = int(2 * (4 * n_embd) / 3)
        self.w1 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, n_embd, bias=False)
        self.w3 = nn.Linear(n_embd, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = RMSNorm(n_embd)
        self.ffwd = SwiGLUFeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class VectoraGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)
        for name, param in self.named_parameters():
            if name.endswith("c_proj.weight") or name.endswith("w2.weight"):
                nn.init.normal_(param, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B * T, -1), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.7, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = VectoraGPT(vocab_size).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count:,} (~{param_count / 1e6:.1f}M)")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
    foreach=False,
)

def get_lr(step):
    if step < warmup_iters:
        return learning_rate * (step + 1) / warmup_iters
    decay_ratio = min(1.0, (step - warmup_iters) / max(1, max_iters - warmup_iters))
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out
@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []

    for _ in range(eval_iters):
        xb, yb = get_batch("val")
        _, loss = model(xb, yb)
        losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)
print(f"Training started for {max_iters:,} steps...")

start_time = time.time()
model.train()

# Initial evaluation before training
initial_losses = estimate_loss()

print(
    f"step {0:5d}/{max_iters} | "
    f"train: {initial_losses['train']:.4f} | "
    f"val: {initial_losses['val']:.4f} | "
    f"lr: {get_lr(0):.2e} | "
    f"tokens: {0:,} | "
    f"time: {time.time() - start_time:.1f}s"
)

for step in range(max_iters):

    lr = get_lr(step)

    for group in optimizer.param_groups:
        group["lr"] = lr

    optimizer.zero_grad(set_to_none=True)

    total_loss = 0.0

    # Gradient accumulation
    for micro_step in range(accumulation_steps):
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)

        total_loss += loss.item()

        (loss / accumulation_steps).backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    # Calculate metrics
    average_loss = total_loss / accumulation_steps
    val_loss = estimate_val_loss()

    tokens_seen = (
        (step + 1) *
        batch_size *
        accumulation_steps *
        block_size
    )

    elapsed = time.time() - start_time

    # Print every iteration
    print(
        f"step {step + 1:5d}/{max_iters} | "
        f"train: {average_loss:.4f} | "
        f"val: {val_loss:.4f} | "
        f"lr: {lr:.2e} | "
        f"tokens: {tokens_seen:,} | "
        f"time: {elapsed:.1f}s"
    )

    # Save checkpoints
    if (step + 1) % checkpoint_interval == 0:
        ckpt_name = f"vectora_step_{step + 1}.pt"

        torch.save({
            "step": step + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": val_loss,
        }, ckpt_name)

        print(f"  [Saved checkpoint: {ckpt_name}]")

    
from safetensors.torch import save_file

save_file(
    model.state_dict(),
    "model.safetensors"
)
print("Training complete!")
torch.save(model.state_dict(), "vectora_final.pt")
print("Saved final model to vectora_final.pt")


Backend: DirectML | Device: privateuseone:0
BERT Vocab size: 50,000
Loaded Train tokens: 170,607,031 | Validation tokens: 18,956,530
Model parameters: 28,745,984 (~28.7M)
Training started for 50,000 steps...
step     0/50000 | train: 10.8573 | val: 10.8236 | lr: 5.00e-07 | tokens: 0 | time: 0.7s
step     1/50000 | train: 10.8781 | val: 10.9007 | lr: 5.00e-07 | tokens: 1,024 | time: 6.5s
step     2/50000 | train: 10.8613 | val: 10.8618 | lr: 1.00e-06 | tokens: 2,048 | time: 12.7s
step     3/50000 | train: 10.8904 | val: 10.8723 | lr: 1.50e-06 | tokens: 3,072 | time: 19.5s
step     4/50000 | train: 10.8894 | val: 10.8375 | lr: 2.00e-06 | tokens: 4,096 | time: 26.1s
step     5/50000 | train: 10.8440 | val: 10.8748 | lr: 2.50e-06 | tokens: 5,120 | time: 32.5s
step     6/50000 | train: 10.8698 | val: 10.8634 | lr: 3.00e-06 | tokens: 6,144 | time: 39.0s
step     7/50000 | train: 10.8720 | val: 10.8409 | lr: 3.50e-06 | tokens: 7,168 | time: 45.5s
step     8/50000 | train: 10.8668 | val: 10.88